# CLVS-ML: Analisis Komparatif Algoritma Machine Learning untuk Sistem Penilaian Kredit Pinjaman Konsumer

## Eksperimen dengan Data Sintetik 10.000 Nasabah Indonesia

---

### Abstrak

Penelitian ini menyajikan eksperimen komparatif tiga algoritma *machine learning* — **Logistic Regression**, **Random Forest**, dan **XGBoost** — untuk masalah klasifikasi multiclass penilaian kredit konsumer di konteks perbankan Indonesia. Dataset sintetik 10.000 nasabah dengan distribusi statistik realistis digunakan, mencakup 36 fitur demografis, keuangan, dan riwayat kredit. Sistem mengklasifikasikan nasabah ke dalam tiga kategori risiko kredit: **Lancar**, **Perhatian**, dan **Macet**.

**Kontribusi Utama:**
- Generasi data sintetik dengan korelasi realistis antara fitur keuangan dan risiko kredit
- Benchmarking komparatif tiga algoritma klasifikasi dengan metrik komprehensif (Accuracy, F1-Macro, Precision, Recall)
- Visualisasi bergaya artikel ilmiah menggunakan Altair 5
- Analisis feature importance untuk interpretabilitas dan transparansi model

---

| Metadata | Nilai |
|---|---|
| **Tanggal Eksperimen** | 7 Maret 2026 |
| **Platform** | Google Colab (Python 3.x) |
| **scikit-learn** | 1.6.1 |
| **XGBoost** | 2.x |
| **Altair** | >= 5.0 |
| **Dataset** | Sintetik — 10.000 nasabah, 36 fitur |
| **Penulis** | [Placeholder] |

In [ ]:
!pip install -q xgboost altair "vegafusion[embed]>=1.5.0" "vl-convert-python>=1.6.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 35.5 MB/s eta 0:00:00


In [14]:
import warnings
import os
import time

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import altair as alt
import joblib
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from xgboost import XGBClassifier

# Konfigurasi Altair untuk Google Colab
alt.data_transformers.enable('vegafusion')

# --- Custom Research Theme ---
def research_theme():
    return {
        'config': {
            'view': {'stroke': 'transparent'},
            'axis': {
                'labelFont': 'serif',
                'titleFont': 'serif',
                'labelFontSize': 17,
                'titleFontSize': 17,
                'grid': True,
                'gridColor': '#e0e0e0'
            },
            'legend': {
                'labelFont': 'serif',
                'titleFont': 'serif',
                'labelFontSize': 17,
                'titleFontSize': 17,
            },
            'title': {'font': 'serif', 'fontSize': 17, 'fontWeight': 'bold'},
            'bar': {'cornerRadiusTopLeft': 2, 'cornerRadiusTopRight': 2},
            'text': {'fontSize': 16},
        }
    }

alt.themes.register('research_theme', research_theme)
alt.themes.enable('research_theme')

# --- Konstanta & Konfigurasi ---
RANDOM_STATE = 42
TEST_SIZE    = 0.2

CATEGORICAL_COLS = [
    'jenis_kelamin', 'status_pernikahan', 'pasangan_bekerja',
    'pendidikan', 'sektor_pekerjaan', 'level_pekerjaan', 'risiko_pekerjaan',
    'status_rumah', 'tujuan_pinjaman', 'kategori_tujuan',
    'riwayat_kredit', 'urban_rural', 'wilayah_risiko',
]

NUMERICAL_COLS = [
    'umur', 'lama_bekerja_tahun', 'lama_di_pekerjaan_sekarang',
    'penghasilan_bulanan', 'penghasilan_tambahan', 'penghasilan_pasangan',
    'total_penghasilan', 'jumlah_tanggungan', 'jumlah_anak',
    'jumlah_pinjaman_diajukan', 'sisa_pinjaman_lain', 'total_eksposur_kredit',
    'tenor_bulan', 'angsuran_bulanan', 'debt_to_income_ratio',
    'loan_to_income_ratio', 'saldo_tabungan', 'nilai_aset',
    'rasio_aset_utang', 'jumlah_kredit_sebelumnya', 'jumlah_tunggakan',
    'hari_tunggakan_terlama', 'lama_tinggal',
]

TARGET_COL = 'status_kredit'

# Palet warna riset
COLOR_SCALE = alt.Scale(
    domain=['Lancar', 'Perhatian', 'Macet'],
    range=['#2196F3', '#FF9800', '#F44336']
)
MODEL_COLOR_SCALE = alt.Scale(
    domain=['Logistic Regression', 'Random Forest', 'XGBoost'],
    range=['#5C85D6', '#4CAF50', '#E53935']
)

print('Semua library berhasil diimport.')
print(f'  numpy   : {np.__version__}')
print(f'  pandas  : {pd.__version__}')
print(f'  altair  : {alt.__version__}')
print(f'  xgboost : {xgb.__version__}')

Semua library berhasil diimport.
  numpy   : 2.0.2
  pandas  : 2.2.2
  altair  : 5.5.0
  xgboost : 3.2.0


## 1. Pendahuluan & Latar Belakang

### 1.1 Credit Scoring dalam Konteks Perbankan Indonesia

Penilaian kredit (*credit scoring*) merupakan proses statistik yang digunakan lembaga keuangan untuk mengevaluasi kelayakan kredit calon debitur. Dalam ekosistem perbankan Indonesia, risiko kredit bermasalah (Non-Performing Loan/NPL) menjadi tantangan signifikan yang mempengaruhi profitabilitas dan stabilitas institusi keuangan.

Otoritas Jasa Keuangan (OJK) mengklasifikasikan kualitas kredit ke dalam lima kategori, namun untuk keperluan penelitian ini disederhanakan menjadi tiga kelas:

| Kelas | Deskripsi | Distribusi Target |
|---|---|---|
| **Lancar** | Pembayaran tepat waktu, tidak ada tunggakan | ~55% |
| **Perhatian** | Tunggakan 1-90 hari, perlu pemantauan | ~35% |
| **Macet** | Tunggakan >90 hari, risiko gagal bayar tinggi | ~10% |

### 1.2 Motivasi Penelitian

Deteksi dini risiko kredit memungkinkan:
- Pengambilan keputusan kredit yang lebih objektif dan konsisten
- Pengurangan NPL melalui identifikasi debitur berisiko sejak dini
- Alokasi sumber daya pengawasan yang lebih efisien

### 1.3 Ringkasan Metodologi

```
Dataset Sintetik 10k  →  EDA & Eksplorasi  →  Preprocessing
        ↓                                           ↓
  Benchmarking  ←──  Evaluasi  ←──  Training (LR, RF, XGBoost)
        ↓
  Analisis Feature Importance & Kesimpulan
```

Dataset dirancang dengan korelasi realistis: nasabah dengan DTI tinggi, riwayat kredit buruk, dan tunggakan banyak cenderung diklasifikasikan sebagai **Macet**.

In [15]:
# ============================================================
# Generasi Data Sintetik 10.000 Nasabah
# ============================================================

CAT_DIST = {
    'jenis_kelamin':    {'P': 0.500, 'L': 0.500},
    'status_pernikahan':{'Menikah': 0.550, 'Single': 0.450},
    'pasangan_bekerja': {'Tidak': 0.619, 'Ya': 0.381},
    'pendidikan':       {'SMA': 0.363, 'S1': 0.345, 'Diploma': 0.197, 'S2': 0.095},
    'sektor_pekerjaan': {'Formal': 0.496, 'Informal': 0.304, 'Wirausaha': 0.200},
    'level_pekerjaan':  {'Staff': 0.598, 'Supervisor': 0.304, 'Manager': 0.098},
    'risiko_pekerjaan': {'Low': 0.495, 'Medium': 0.356, 'High': 0.149},
    'status_rumah':     {'Milik': 0.452, 'KPR': 0.337, 'Kontrak': 0.211},
    'tujuan_pinjaman':  {'Konsumtif': 0.601, 'Produktif': 0.399},
    'kategori_tujuan':  {'Renovasi': 0.255, 'Multiguna': 0.255, 'Kendaraan': 0.247, 'Usaha': 0.244},
    'riwayat_kredit':   {'Baik': 0.611, 'Cukup': 0.293, 'Buruk': 0.096},
    'urban_rural':      {'Urban': 0.645, 'Rural': 0.355},
    'wilayah_risiko':   {'Low': 0.500, 'Medium': 0.298, 'High': 0.202},
}

TARGET_DIST = {'Lancar': 0.554, 'Perhatian': 0.349, 'Macet': 0.097}


def _sample_cat(col, n, rng):
    cats  = list(CAT_DIST[col].keys())
    probs = np.array(list(CAT_DIST[col].values()), dtype=np.float64)
    probs /= probs.sum()
    return rng.choice(cats, size=n, p=probs)


def _make_numerical_segment(status, n, rng):
    def clip_int(arr, lo, hi):
        return np.clip(np.round(arr).astype(int), lo, hi)

    def clip_float(arr, lo, hi):
        return np.clip(arr, lo, hi)

    params = {
        'Lancar': dict(
            income_mean=7_500_000, income_std=2_000_000,
            dti_mean=0.15, dti_std=0.08,
            tunggakan_mean=0.1, hari_tung_mean=5,
            kredit_sblm_mean=2.0, saldo_mult=2.5,
        ),
        'Perhatian': dict(
            income_mean=6_000_000, income_std=1_800_000,
            dti_mean=0.30, dti_std=0.10,
            tunggakan_mean=0.5, hari_tung_mean=35,
            kredit_sblm_mean=2.5, saldo_mult=1.5,
        ),
        'Macet': dict(
            income_mean=4_500_000, income_std=1_500_000,
            dti_mean=0.60, dti_std=0.20,
            tunggakan_mean=2.0, hari_tung_mean=65,
            kredit_sblm_mean=3.0, saldo_mult=0.5,
        ),
    }[status]

    income    = clip_float(rng.normal(params['income_mean'], params['income_std'], n), 2e6, 12.5e6)
    add_inc   = clip_float(rng.exponential(800_000, n), 0, 3.7e6)
    sp_inc    = clip_float(rng.exponential(1_500_000, n), 0, 9e6)
    total_inc = income + add_inc + sp_inc

    pinjaman  = clip_float(rng.uniform(5e6, 173e6, n), 5e6, 173e6)
    sisa      = clip_float(rng.uniform(0, 71e6, n), 0, 71e6)
    total_eks = clip_float(pinjaman + sisa, 5e6, 210e6)

    tenor_p   = np.array([0.1, 0.2, 0.3, 0.25, 0.15]); tenor_p /= tenor_p.sum()
    tenor     = clip_int(rng.choice([12, 24, 36, 48, 60], n, p=tenor_p), 12, 60)
    angsuran  = clip_float(pinjaman / tenor, 83_333, 14.5e6)

    dti      = clip_float(rng.normal(params['dti_mean'], params['dti_std'], n), 0.005, 3.0)
    lti      = clip_float(pinjaman / np.maximum(total_inc, 1), 0.28, 38.5)
    saldo    = clip_float(rng.exponential(params['saldo_mult'] * 1e7, n), 0, 48.7e6)
    aset     = clip_float(rng.uniform(10e6, 313e6, n), 10e6, 313e6)
    rasio_au = clip_float(rng.exponential(1.8, n), 0.067, 62.0)

    tungg   = clip_int(rng.poisson(params['tunggakan_mean'], n), 0, 4)
    hari_t  = clip_int(rng.exponential(params['hari_tung_mean'], n), 0, 89)
    kr_sblm = clip_int(rng.poisson(params['kredit_sblm_mean'], n), 0, 5)

    umur         = clip_int(rng.uniform(21, 59, n), 21, 59)
    lama_kerja   = clip_int(rng.uniform(0, 29, n), 0, 29)
    lama_skrg    = clip_int(rng.uniform(0, 14, n), 0, 14)
    tanggungan   = clip_int(rng.poisson(2.0, n), 0, 4)
    anak         = clip_int(np.minimum(tanggungan, rng.poisson(1.5, n)), 0, 3)
    lama_tinggal = clip_int(rng.uniform(1, 19, n), 1, 19)

    return pd.DataFrame({
        'umur':                       umur,
        'lama_bekerja_tahun':         lama_kerja,
        'lama_di_pekerjaan_sekarang': lama_skrg,
        'penghasilan_bulanan':        income,
        'penghasilan_tambahan':       add_inc,
        'penghasilan_pasangan':       sp_inc,
        'total_penghasilan':          total_inc,
        'jumlah_tanggungan':          tanggungan,
        'jumlah_anak':                anak,
        'jumlah_pinjaman_diajukan':   pinjaman,
        'sisa_pinjaman_lain':         sisa,
        'total_eksposur_kredit':      total_eks,
        'tenor_bulan':                tenor,
        'angsuran_bulanan':           angsuran,
        'debt_to_income_ratio':       dti,
        'loan_to_income_ratio':       lti,
        'saldo_tabungan':             saldo,
        'nilai_aset':                 aset,
        'rasio_aset_utang':           rasio_au,
        'jumlah_kredit_sebelumnya':   kr_sblm,
        'jumlah_tunggakan':           tungg,
        'hari_tunggakan_terlama':     hari_t,
        'lama_tinggal':               lama_tinggal,
    })


def generate(n=10_000, seed=42):
    rng = np.random.default_rng(seed)

    statuses = list(TARGET_DIST.keys())
    probs    = np.array(list(TARGET_DIST.values()), dtype=np.float64)
    probs   /= probs.sum()
    labels   = rng.choice(statuses, size=n, p=probs)

    segments = []
    for status in statuses:
        idx   = np.where(labels == status)[0]
        n_seg = len(idx)
        if n_seg == 0:
            continue

        num_df = _make_numerical_segment(status, n_seg, rng)

        for col in CAT_DIST:
            num_df[col] = _sample_cat(col, n_seg, rng)

        def _riwayat_p(*weights):
            p = np.array(weights, dtype=np.float64)
            p /= p.sum()
            return p

        if status == 'Macet':
            num_df['riwayat_kredit'] = rng.choice(
                ['Buruk', 'Cukup', 'Baik'], n_seg, p=_riwayat_p(0.60, 0.30, 0.10)
            )
        elif status == 'Perhatian':
            num_df['riwayat_kredit'] = rng.choice(
                ['Buruk', 'Cukup', 'Baik'], n_seg, p=_riwayat_p(0.15, 0.50, 0.35)
            )
        else:
            num_df['riwayat_kredit'] = rng.choice(
                ['Buruk', 'Cukup', 'Baik'], n_seg, p=_riwayat_p(0.02, 0.18, 0.80)
            )

        num_df['status_kredit'] = status
        segments.append(num_df)

    df_out = pd.concat(segments, ignore_index=True)
    df_out = df_out.sample(frac=1, random_state=seed).reset_index(drop=True)

    col_order = [
        'umur', 'lama_bekerja_tahun', 'lama_di_pekerjaan_sekarang',
        'penghasilan_bulanan', 'penghasilan_tambahan', 'penghasilan_pasangan',
        'total_penghasilan', 'jumlah_tanggungan', 'jumlah_anak',
        'jumlah_pinjaman_diajukan', 'sisa_pinjaman_lain', 'total_eksposur_kredit',
        'tenor_bulan', 'angsuran_bulanan', 'debt_to_income_ratio', 'loan_to_income_ratio',
        'saldo_tabungan', 'nilai_aset', 'rasio_aset_utang',
        'jumlah_kredit_sebelumnya', 'jumlah_tunggakan', 'hari_tunggakan_terlama',
        'lama_tinggal', 'jenis_kelamin', 'status_pernikahan', 'pasangan_bekerja',
        'pendidikan', 'sektor_pekerjaan', 'level_pekerjaan', 'risiko_pekerjaan',
        'status_rumah', 'tujuan_pinjaman', 'kategori_tujuan', 'riwayat_kredit',
        'urban_rural', 'wilayah_risiko', 'status_kredit',
    ]
    return df_out[col_order]


# --- Generate & Simpan ---
print('Generating 10.000 data sintetik...')
df = generate(10_000, seed=42)

print(f'Shape    : {df.shape[0]:,} baris x {df.shape[1]} kolom')
print(f'Kolom    : {list(df.columns)}')
print('\nDistribusi Kelas Target:')
for cls, cnt in df[TARGET_COL].value_counts().items():
    pct = cnt / len(df) * 100
    print(f'  {cls:12s}: {cnt:5,d} ({pct:.1f}%)')

df.to_csv('synthetic_credit_10k.csv', index=False)
print('\nFile tersimpan: synthetic_credit_10k.csv')

Generating 10.000 data sintetik...
Shape    : 10,000 baris x 37 kolom
Kolom    : ['umur', 'lama_bekerja_tahun', 'lama_di_pekerjaan_sekarang', 'penghasilan_bulanan', 'penghasilan_tambahan', 'penghasilan_pasangan', 'total_penghasilan', 'jumlah_tanggungan', 'jumlah_anak', 'jumlah_pinjaman_diajukan', 'sisa_pinjaman_lain', 'total_eksposur_kredit', 'tenor_bulan', 'angsuran_bulanan', 'debt_to_income_ratio', 'loan_to_income_ratio', 'saldo_tabungan', 'nilai_aset', 'rasio_aset_utang', 'jumlah_kredit_sebelumnya', 'jumlah_tunggakan', 'hari_tunggakan_terlama', 'lama_tinggal', 'jenis_kelamin', 'status_pernikahan', 'pasangan_bekerja', 'pendidikan', 'sektor_pekerjaan', 'level_pekerjaan', 'risiko_pekerjaan', 'status_rumah', 'tujuan_pinjaman', 'kategori_tujuan', 'riwayat_kredit', 'urban_rural', 'wilayah_risiko', 'status_kredit']

Distribusi Kelas Target:
  Lancar      : 5,573 (55.7%)
  Perhatian   : 3,473 (34.7%)
  Macet       :   954 (9.5%)

File tersimpan: synthetic_credit_10k.csv


## 2. Eksplorasi & Informasi Dataset

In [16]:
# --- 2.1 Statistik Deskriptif ---
print('=== Informasi Dataset ===')
print(f'Shape  : {df.shape}')
print(f'Memory : {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
print(f'\nTipe data:')
print(df.dtypes.value_counts().to_string())

print('\n=== Missing Values ===')
missing = df.isnull().sum()
if missing.any():
    print(missing[missing > 0].to_string())
else:
    print('Tidak ada missing values — dataset bersih.')

print('\n=== Statistik Deskriptif Fitur Numerik Kunci ===')
KEY_NUM = ['debt_to_income_ratio', 'loan_to_income_ratio', 'penghasilan_bulanan',
           'angsuran_bulanan', 'jumlah_tunggakan', 'hari_tunggakan_terlama',
           'saldo_tabungan', 'nilai_aset']
display(df[KEY_NUM].describe().round(3))

# --- Chart 1: Donut chart distribusi kelas target ---
status_counts = df[TARGET_COL].value_counts().reset_index()
status_counts.columns = ['Status Kredit', 'Jumlah']
status_counts['Persentase'] = (status_counts['Jumlah'] / len(df) * 100).round(1)

print('\n=== Data Chart 1: Distribusi Kelas Target ===')
for _, row in status_counts.iterrows():
    print(f'  {row["Status Kredit"]:12s}: {row["Jumlah"]:>5,} nasabah  ({row["Persentase"]:.1f}%)')
print(f'  {"Total":12s}: {status_counts["Jumlah"].sum():>5,} nasabah')

chart1 = alt.Chart(status_counts).mark_arc(innerRadius=60, outerRadius=130).encode(
    theta=alt.Theta('Jumlah:Q'),
    color=alt.Color('Status Kredit:N', scale=COLOR_SCALE,
                    legend=alt.Legend(title='Status Kredit')),
    tooltip=[
        alt.Tooltip('Status Kredit:N'),
        alt.Tooltip('Jumlah:Q'),
        alt.Tooltip('Persentase:Q', format='.1f', title='Persen (%)')
    ]
).properties(
    title='Distribusi Kelas Target — Status Kredit',
    width=320, height=280
)
display(chart1)

# --- Chart 2: Heatmap korelasi fitur numerik kunci ---
LABEL_SHORT = {
    'debt_to_income_ratio':   'DTI Ratio',
    'loan_to_income_ratio':   'LTI Ratio',
    'penghasilan_bulanan':    'Penghasilan',
    'angsuran_bulanan':       'Angsuran',
    'jumlah_tunggakan':       'Tunggakan',
    'hari_tunggakan_terlama': 'Hari Tung.',
    'saldo_tabungan':         'Tabungan',
    'nilai_aset':             'Aset',
    'total_penghasilan':      'Total Pghsln',
}
NUM_KEY_COLS = list(LABEL_SHORT.keys())

corr_matrix = df[NUM_KEY_COLS].corr()
corr_matrix.index   = [LABEL_SHORT[c] for c in corr_matrix.index]
corr_matrix.columns = [LABEL_SHORT[c] for c in corr_matrix.columns]
corr_data = corr_matrix.reset_index().melt(id_vars='index')
corr_data.columns = ['Fitur A', 'Fitur B', 'Korelasi']

print('\n=== Data Chart 2: Matriks Korelasi Fitur Numerik Kunci ===')
display(corr_matrix.round(3))

chart2 = alt.Chart(corr_data).mark_rect().encode(
    x=alt.X('Fitur A:N', title=None, axis=alt.Axis(labelAngle=-30)),
    y=alt.Y('Fitur B:N', title=None),
    color=alt.Color('Korelasi:Q',
                    scale=alt.Scale(scheme='redblue', domain=[-1, 1]),
                    legend=alt.Legend(title='Korelasi')),
    tooltip=[
        alt.Tooltip('Fitur A:N'),
        alt.Tooltip('Fitur B:N'),
        alt.Tooltip('Korelasi:Q', format='.3f')
    ]
).properties(
    title='Heatmap Korelasi Fitur Numerik Kunci',
    width=440, height=380
)
chart2

=== Informasi Dataset ===
Shape  : (10000, 37)
Memory : 9226.9 KB

Tipe data:
object     14
float64    13
int64      10

=== Missing Values ===
Tidak ada missing values — dataset bersih.

=== Statistik Deskriptif Fitur Numerik Kunci ===


,debt_to_income_ratio,loan_to_income_ratio,penghasilan_bulanan,angsuran_bulanan,jumlah_tunggakan,hari_tunggakan_terlama,saldo_tabungan,nilai_aset
count,10000.000,10000.000,1.000000e+04,1.000000e+04,10000.000,10000.000,1.000000e+04,1.000000e+04
mean,0.245,10.973,6.681519e+06,2.898142e+06,0.413,19.027,1.744465e+07,1.620265e+08
std,0.170,7.297,2.108323e+06,2.466458e+06,0.805,25.343,1.532664e+07,8.762831e+07
min,0.005,0.350,2.000000e+06,8.406695e+04,0.000,0.000,8.013740e+02,1.002152e+07
25%,0.132,5.210,5.193412e+06,1.243810e+06,0.000,3.000,4.825334e+06,8.692666e+07
50%,0.210,9.998,6.634262e+06,2.360319e+06,0.000,7.000,1.247087e+07,1.612913e+08
75%,0.316,15.226,8.089565e+06,3.618397e+06,1.000,24.000,2.672382e+07,2.388456e+08
max,1.214,38.500,1.250000e+07,1.440307e+07,4.000,89.000,4.870000e+07,3.129305e+08



=== Data Chart 1: Distribusi Kelas Target ===
  Lancar      : 5,573 nasabah  (55.7%)
  Perhatian   : 3,473 nasabah  (34.7%)
  Macet       :   954 nasabah  (9.5%)
  Total       : 10,000 nasabah


alt.Chart(...)


=== Data Chart 2: Matriks Korelasi Fitur Numerik Kunci ===


,DTI Ratio,LTI Ratio,Penghasilan,Angsuran,Tunggakan,Hari Tung.,Tabungan,Aset,Total Pghsln
DTI Ratio,1.000,0.158,-0.362,-0.014,0.522,0.483,-0.269,-0.001,-0.282
LTI Ratio,0.158,1.000,-0.399,0.521,0.119,0.128,-0.067,-0.000,-0.476
Penghasilan,-0.362,-0.399,1.000,0.008,-0.278,-0.301,0.164,-0.004,0.783
Angsuran,-0.014,0.521,0.008,1.000,-0.007,0.001,0.000,-0.001,-0.014
Tunggakan,0.522,0.119,-0.278,-0.007,1.000,0.354,-0.212,-0.006,-0.201
Hari Tung.,0.483,0.128,-0.301,0.001,0.354,1.000,-0.212,-0.004,-0.238
Tabungan,-0.269,-0.067,0.164,0.000,-0.212,-0.212,1.000,-0.006,0.129
Aset,-0.001,-0.000,-0.004,-0.001,-0.006,-0.004,-0.006,1.000,-0.010
Total Pghsln,-0.282,-0.476,0.783,-0.014,-0.201,-0.238,0.129,-0.010,1.000


alt.Chart(...)

In [17]:
# --- Chart 3: Boxplot debt_to_income_ratio per kelas ---
print('=== Data Chart 3: Statistik DTI Ratio per Kelas ===')
dti_stats = df.groupby('status_kredit')['debt_to_income_ratio'].describe().round(4)
display(dti_stats)

chart3 = alt.Chart(df).mark_boxplot(extent='min-max', size=30).encode(
    x=alt.X('status_kredit:N', title='Status Kredit',
            axis=alt.Axis(labelAngle=0),
            sort=['Lancar', 'Perhatian', 'Macet']),
    y=alt.Y('debt_to_income_ratio:Q', title='Debt-to-Income Ratio',
            scale=alt.Scale(domain=[0, 1.5])),
    color=alt.Color('status_kredit:N', scale=COLOR_SCALE, legend=None)
).properties(
    title='Distribusi Debt-to-Income Ratio per Kelas',
    width=300, height=200
)
display(chart3)

# --- Chart 4: Grouped bar riwayat kredit per kelas ---
riwayat_dist = (
    df.groupby(['status_kredit', 'riwayat_kredit'])
    .size()
    .reset_index(name='Jumlah')
)
riwayat_dist.columns = ['Status Kredit', 'Riwayat Kredit', 'Jumlah']

print('\n=== Data Chart 4: Distribusi Riwayat Kredit per Status ===')
riwayat_pivot = riwayat_dist.pivot(index='Riwayat Kredit', columns='Status Kredit', values='Jumlah')
riwayat_pivot = riwayat_pivot[['Lancar', 'Perhatian', 'Macet']]
display(riwayat_pivot)

chart4 = alt.Chart(riwayat_dist).mark_bar().encode(
    x=alt.X('Riwayat Kredit:N', title='Riwayat Kredit',
            sort=['Baik', 'Cukup', 'Buruk']),
    y=alt.Y('Jumlah:Q', title='Jumlah Nasabah'),
    color=alt.Color('Status Kredit:N', scale=COLOR_SCALE,
                    legend=alt.Legend(title='Status Kredit')),
    xOffset=alt.XOffset('Status Kredit:N'),
    tooltip=['Status Kredit:N', 'Riwayat Kredit:N', 'Jumlah:Q']
).properties(
    title='Distribusi Riwayat Kredit per Status Kredit',
    width=380, height=300
)
display(chart4)

# --- Chart 5: Scatter penghasilan vs DTI ---
print('\n=== Data Chart 5: Ringkasan Penghasilan vs DTI per Kelas (sampel 2.000) ===')
sample_5 = df.sample(2000, random_state=42)
summary_5 = sample_5.groupby('status_kredit').agg(
    Jumlah=('penghasilan_bulanan', 'count'),
    Mean_Penghasilan=('penghasilan_bulanan', 'mean'),
    Mean_DTI=('debt_to_income_ratio', 'mean'),
    Median_Penghasilan=('penghasilan_bulanan', 'median'),
    Median_DTI=('debt_to_income_ratio', 'median'),
).round(2)
display(summary_5)

chart5 = alt.Chart(sample_5).mark_circle(
    size=40, opacity=0.45
).encode(
    x=alt.X('penghasilan_bulanan:Q', title='Penghasilan Bulanan (Rp)',
            axis=alt.Axis(format='~s')),
    y=alt.Y('debt_to_income_ratio:Q', title='Debt-to-Income Ratio',
            scale=alt.Scale(domain=[0, 1.5])),
    color=alt.Color('status_kredit:N', scale=COLOR_SCALE,
                    legend=alt.Legend(title='Status Kredit')),
    tooltip=[
        alt.Tooltip('status_kredit:N', title='Status'),
        alt.Tooltip('penghasilan_bulanan:Q', format=',.0f', title='Penghasilan'),
        alt.Tooltip('debt_to_income_ratio:Q', format='.3f', title='DTI')
    ]
).properties(
    title='Penghasilan Bulanan vs Debt-to-Income Ratio (Sampel 2.000)',
    width=500, height=500
)
chart5

=== Data Chart 3: Statistik DTI Ratio per Kelas ===


,count,mean,std,min,25%,50%,75%,max
status_kredit,,,,,,,,
Lancar,5573.0,0.1499,0.0769,0.005,0.0946,0.1481,0.2024,0.4768
Macet,954.0,0.6037,0.2036,0.005,0.4694,0.6074,0.7431,1.2136
Perhatian,3473.0,0.2997,0.0995,0.005,0.2333,0.3008,0.3662,0.6350


alt.Chart(...)


=== Data Chart 4: Distribusi Riwayat Kredit per Status ===


Status Kredit,Lancar,Perhatian,Macet
Riwayat Kredit,,,
Baik,4462,1216,99
Buruk,107,518,577
Cukup,1004,1739,278


alt.Chart(...)


=== Data Chart 5: Ringkasan Penghasilan vs DTI per Kelas (sampel 2.000) ===


,Jumlah,Mean_Penghasilan,Mean_DTI,Median_Penghasilan,Median_DTI
status_kredit,,,,,
Lancar,1111,7438793.72,0.15,7441890.73,0.15
Macet,203,4438959.64,0.60,4515444.34,0.61
Perhatian,686,5976217.26,0.30,5975679.92,0.30


alt.Chart(...)

## 3. Pra-pemrosesan Data

XGBoost dan Random Forest adalah model berbasis pohon keputusan — **tidak memerlukan scaling fitur**. Hanya transformasi berikut yang diperlukan:

1. **LabelEncoder** untuk 13 fitur kategorikal → integer encoding
2. **LabelEncoder** untuk target → integer encoding (0, 1, 2)
3. **Train/Test Split** stratified 80/20

> **Catatan:** Logistic Regression membutuhkan StandardScaler karena sensitif terhadap skala fitur. Scaling hanya diterapkan untuk model LR.

In [18]:
# --- Encode Kategorikal ---
encoders = {}
df_enc = df.copy()
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))
    encoders[col] = le

# --- Encode Target ---
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(df_enc[TARGET_COL])
CLASSES = list(target_encoder.classes_)
print(f'Kelas target (terurut): {CLASSES}')

# --- Feature matrix ---
FEATURE_COLS = NUMERICAL_COLS + CATEGORICAL_COLS
X = df_enc[FEATURE_COLS].values

# --- Train/Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print(f'Train: {X_train.shape[0]:,} baris | Test: {X_test.shape[0]:,} baris')

# --- Tabel distribusi kelas ---
print('\nDistribusi kelas di Train:')
for cls_idx, cls_name in enumerate(CLASSES):
    cnt = (y_train == cls_idx).sum()
    print(f'  {cls_name:12s}: {cnt:,}')

print('Distribusi kelas di Test:')
for cls_idx, cls_name in enumerate(CLASSES):
    cnt = (y_test == cls_idx).sum()
    print(f'  {cls_name:12s}: {cnt:,}')

# --- Chart 6: Bar distribusi train vs test ---
class_map = dict(enumerate(CLASSES))
train_dist = pd.Series(y_train).map(class_map).value_counts().reset_index()
train_dist.columns = ['Status Kredit', 'Jumlah']
train_dist['Split'] = 'Train'

test_dist = pd.Series(y_test).map(class_map).value_counts().reset_index()
test_dist.columns = ['Status Kredit', 'Jumlah']
test_dist['Split'] = 'Test'

split_df = pd.concat([train_dist, test_dist], ignore_index=True)

print('\n=== Data Chart 6: Distribusi Kelas Train vs Test ===')
split_pivot = split_df.pivot(index='Status Kredit', columns='Split', values='Jumlah')
split_pivot = split_pivot[['Train', 'Test']]
split_pivot['Total'] = split_pivot.sum(axis=1)
display(split_pivot)

chart6 = alt.Chart(split_df).mark_bar().encode(
    x=alt.X('Split:N', title=None, axis=alt.Axis(labelAngle=0)),
    y=alt.Y('Jumlah:Q', title='Jumlah Nasabah'),
    color=alt.Color('Status Kredit:N', scale=COLOR_SCALE,
                    legend=alt.Legend(title='Status Kredit')),
    xOffset=alt.XOffset('Status Kredit:N'),
    tooltip=['Split:N', 'Status Kredit:N', 'Jumlah:Q']
).properties(
    title='Distribusi Kelas: Train vs Test Split (80/20 Stratified)',
    width=340, height=300
)
chart6

Kelas target (terurut): ['Lancar', 'Macet', 'Perhatian']
Train: 8,000 baris | Test: 2,000 baris

Distribusi kelas di Train:
  Lancar      : 4,459
  Macet       : 763
  Perhatian   : 2,778
Distribusi kelas di Test:
  Lancar      : 1,114
  Macet       : 191
  Perhatian   : 695

=== Data Chart 6: Distribusi Kelas Train vs Test ===


Split,Train,Test,Total
Status Kredit,,,
Lancar,4459,1114,5573
Macet,763,191,954
Perhatian,2778,695,3473


alt.Chart(...)

## 4. Pelatihan Model

Tiga model dilatih dan dibandingkan:

| Model | Kelebihan | Catatan |
|---|---|---|
| **Logistic Regression** | Interpretable, baseline klasik | Butuh scaling, asumsi linearitas |
| **Random Forest** | Robust, tidak butuh scaling | Ensemble tree, bisa overfit |
| **XGBoost** | State-of-the-art tabular, built-in regularisasi | Gradient boosting, memori lebih besar |

In [19]:
# ============================================================
# Training XGBoost (dengan Early Stopping)
# ============================================================

# XGBoost 2.x sklearn API:
#   - use_label_encoder  → dihapus dari XGB_PARAMS
#   - early_stopping_rounds → pindah ke constructor, BUKAN fit()
#   - callbacks= di fit() → dihapus

XGB_PARAMS = dict(
    n_estimators        = 500,
    max_depth           = 6,
    learning_rate       = 0.05,
    subsample           = 0.8,
    colsample_bytree    = 0.8,
    min_child_weight    = 5,
    gamma               = 0.1,
    reg_alpha           = 0.1,
    reg_lambda          = 1.0,
    objective           = 'multi:softprob',
    eval_metric         = 'mlogloss',
    early_stopping_rounds = 20,   # <-- constructor, bukan fit()
    random_state        = RANDOM_STATE,
    n_jobs              = -1,
)

num_classes = len(CLASSES)
xgb_model = XGBClassifier(**XGB_PARAMS, num_class=num_classes)

print(f'Training XGBoost (max {XGB_PARAMS["n_estimators"]} trees, early stop=20)...')
t0 = time.time()
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False
)
t_xgb = time.time() - t0

print(f'Training selesai dalam {t_xgb:.1f} detik')
print(f'Best iteration   : {xgb_model.best_iteration}')
print(f'Best val mlogloss: {xgb_model.best_score:.5f}')

Training XGBoost (max 500 trees, early stop=20)...
Training selesai dalam 3.6 detik
Best iteration   : 172
Best val mlogloss: 0.24000


In [20]:
# ============================================================
# Training Logistic Regression & Random Forest (Baseline)
# ============================================================

# LR butuh scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Logistic Regression
print('Training Logistic Regression...')
t0 = time.time()
lr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_model.fit(X_train_scaled, y_train)
t_lr = time.time() - t0

# Random Forest
print('Training Random Forest...')
t0 = time.time()
rf_model = RandomForestClassifier(n_estimators=200, max_depth=8,
                                   random_state=RANDOM_STATE, n_jobs=-1)
rf_model.fit(X_train, y_train)
t_rf = time.time() - t0

# --- Fungsi metrik ---
def get_metrics(model, X, y, name):
    y_pred = model.predict(X)
    return {
        'Model':            name,
        'Accuracy':         accuracy_score(y, y_pred),
        'F1-Macro':         f1_score(y, y_pred, average='macro'),
        'Precision-Macro':  precision_score(y, y_pred, average='macro'),
        'Recall-Macro':     recall_score(y, y_pred, average='macro'),
    }

results_lr  = get_metrics(lr_model,  X_test_scaled, y_test, 'Logistic Regression')
results_rf  = get_metrics(rf_model,  X_test,        y_test, 'Random Forest')
results_xgb = get_metrics(xgb_model, X_test,        y_test, 'XGBoost')

metrics_df = pd.DataFrame([results_lr, results_rf, results_xgb])
metrics_df = metrics_df.set_index('Model')

print(f'\nLR   : {t_lr:.1f}s | Accuracy={results_lr["Accuracy"]:.4f} | F1-Macro={results_lr["F1-Macro"]:.4f}')
print(f'RF   : {t_rf:.1f}s | Accuracy={results_rf["Accuracy"]:.4f} | F1-Macro={results_rf["F1-Macro"]:.4f}')
print(f'XGB  : {t_xgb:.1f}s | Accuracy={results_xgb["Accuracy"]:.4f} | F1-Macro={results_xgb["F1-Macro"]:.4f}')

display(metrics_df.style.format('{:.4f}').background_gradient(cmap='Greens', axis=0))

Training Logistic Regression...
Training Random Forest...

LR   : 0.3s | Accuracy=0.9085 | F1-Macro=0.8851
RF   : 3.4s | Accuracy=0.9080 | F1-Macro=0.8813
XGB  : 3.6s | Accuracy=0.9085 | F1-Macro=0.8907


,Accuracy,F1-Macro,Precision-Macro,Recall-Macro
Model,,,,
Logistic Regression,0.9085,0.8851,0.9020,0.8706
Random Forest,0.9080,0.8813,0.9146,0.8566
XGBoost,0.9085,0.8907,0.9069,0.8768


## 5. Benchmarking & Perbandingan Model

In [21]:
# --- Chart 7: Grouped bar — perbandingan semua metrik per model ---
metrics_plot = metrics_df.reset_index().melt(
    id_vars='Model', var_name='Metrik', value_name='Nilai'
)

print('=== Data Chart 7: Perbandingan Performa Model ===')
display(metrics_df.round(4))

chart7 = alt.Chart(metrics_plot).mark_bar().encode(
    x=alt.X('Metrik:N', title=None, axis=alt.Axis(labelAngle=-15)),
    y=alt.Y('Nilai:Q', title='Score',
            scale=alt.Scale(domain=[0, 1])),
    color=alt.Color('Model:N', scale=MODEL_COLOR_SCALE,
                    legend=alt.Legend(title='Model')),
    xOffset=alt.XOffset('Model:N'),
    tooltip=[
        alt.Tooltip('Model:N'),
        alt.Tooltip('Metrik:N'),
        alt.Tooltip('Nilai:Q', format='.4f', title='Score')
    ]
).properties(
    title='Perbandingan Performa Model — Benchmarking',
    width=520, height=360
)
display(chart7)

# --- Chart 8: Heatmap F1 per kelas per model ---
def get_per_class_f1(model, X, y, name):
    y_pred    = model.predict(X)
    f1_scores = f1_score(y, y_pred, average=None)
    return pd.DataFrame({
        'Model': name,
        'Kelas': CLASSES,
        'F1':    f1_scores
    })

f1_lr  = get_per_class_f1(lr_model,  X_test_scaled, y_test, 'Logistic Regression')
f1_rf  = get_per_class_f1(rf_model,  X_test,        y_test, 'Random Forest')
f1_xgb = get_per_class_f1(xgb_model, X_test,        y_test, 'XGBoost')
f1_all = pd.concat([f1_lr, f1_rf, f1_xgb], ignore_index=True)

print('\n=== Data Chart 8: F1-Score per Kelas per Model ===')
f1_pivot = f1_all.pivot(index='Model', columns='Kelas', values='F1').round(4)
f1_pivot = f1_pivot[CLASSES]
display(f1_pivot)

base8 = alt.Chart(f1_all).encode(
    x=alt.X('Kelas:N', title='Kelas Target', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('Model:N', title=None)
)
heat8 = base8.mark_rect().encode(
    color=alt.Color('F1:Q',
                    scale=alt.Scale(scheme='greens', domain=[0, 1]),
                    legend=alt.Legend(title='F1 Score')),
    tooltip=['Model:N', 'Kelas:N', alt.Tooltip('F1:Q', format='.4f')]
)
text8 = base8.mark_text(baseline='middle', fontSize=15, fontWeight='bold').encode(
    text=alt.Text('F1:Q', format='.3f'),
    color=alt.condition(
        alt.datum.F1 > 0.6,
        alt.value('white'),
        alt.value('black')
    )
)
chart8 = (heat8 + text8).properties(
    title='Heatmap F1-Score per Kelas per Model',
    width=380, height=220
)
chart8

=== Data Chart 7: Perbandingan Performa Model ===


,Accuracy,F1-Macro,Precision-Macro,Recall-Macro
Model,,,,
Logistic Regression,0.9085,0.8851,0.9020,0.8706
Random Forest,0.9080,0.8813,0.9146,0.8566
XGBoost,0.9085,0.8907,0.9069,0.8768


alt.Chart(...)


=== Data Chart 8: F1-Score per Kelas per Model ===


Kelas,Lancar,Macet,Perhatian
Model,,,
Logistic Regression,0.9436,0.8444,0.8672
Random Forest,0.9439,0.8328,0.8673
XGBoost,0.9404,0.8643,0.8675


alt.LayerChart(...)

In [22]:
# ============================================================
# Evaluasi Detail — XGBoost
# ============================================================

y_pred_xgb = xgb_model.predict(X_test)

print('Classification Report — XGBoost')
print('=' * 58)
report_dict = classification_report(y_test, y_pred_xgb,
                                     target_names=CLASSES,
                                     output_dict=True)
report_df = pd.DataFrame(report_dict).transpose().round(4)
display(report_df.style.background_gradient(cmap='Blues',
        subset=['precision', 'recall', 'f1-score']))

# --- Chart 9: Confusion Matrix Heatmap ---
cm      = confusion_matrix(y_test, y_pred_xgb)
cm_max  = int(cm.max())
cm_data = pd.DataFrame([
    {'Aktual': CLASSES[i], 'Prediksi': CLASSES[j], 'Nilai': int(cm[i, j])}
    for i in range(len(CLASSES))
    for j in range(len(CLASSES))
])

print('\n=== Data Chart 9: Confusion Matrix — XGBoost ===')
cm_df = pd.DataFrame(cm, index=CLASSES, columns=CLASSES)
cm_df.index.name = 'Aktual'
cm_df.columns.name = 'Prediksi'
display(cm_df)
print(f'  Total prediksi benar : {cm.diagonal().sum():,} / {cm.sum():,}')
print(f'  Akurasi              : {cm.diagonal().sum() / cm.sum():.4f}')

base9 = alt.Chart(cm_data).encode(
    x=alt.X('Prediksi:N', title='Prediksi',
            sort=CLASSES, axis=alt.Axis(labelAngle=0)),
    y=alt.Y('Aktual:N', title='Aktual', sort=CLASSES)
)
heat9 = base9.mark_rect().encode(
    color=alt.Color('Nilai:Q',
                    scale=alt.Scale(scheme='blues'),
                    legend=alt.Legend(title='Jumlah'))
)
text9 = base9.mark_text(fontSize=16, fontWeight='bold').encode(
    text=alt.Text('Nilai:Q'),
    color=alt.condition(
        alt.datum.Nilai > cm_max // 2,
        alt.value('white'),
        alt.value('black')
    )
)
chart9 = (heat9 + text9).properties(
    title='Confusion Matrix — XGBoost Credit Scoring',
    width=360, height=300
)
display(chart9)

# --- Chart 10: Training Loss (Train vs Validasi) ---
results_evals = xgb_model.evals_result()
train_loss    = results_evals['validation_0']['mlogloss']
val_loss      = results_evals['validation_1']['mlogloss']
n_rounds      = len(train_loss)
best_round    = xgb_model.best_iteration + 1  # 1-indexed

loss_df = pd.DataFrame({
    'Round':   list(range(1, n_rounds + 1)) * 2,
    'Loss':    train_loss + val_loss,
    'Dataset': ['Train'] * n_rounds + ['Validasi'] * n_rounds
})

print('\n=== Data Chart 10: Training Loss — XGBoost ===')
print(f'  Total boosting rounds : {n_rounds}')
print(f'  Best iteration        : {best_round}')
print(f'  Train loss (awal)     : {train_loss[0]:.6f}')
print(f'  Train loss (akhir)    : {train_loss[-1]:.6f}')
print(f'  Val loss (awal)       : {val_loss[0]:.6f}')
print(f'  Val loss (akhir)      : {val_loss[-1]:.6f}')
print(f'  Val loss (best)       : {val_loss[best_round - 1]:.6f}')

line10 = alt.Chart(loss_df).mark_line(strokeWidth=2).encode(
    x=alt.X('Round:Q', title='Boosting Round'),
    y=alt.Y('Loss:Q', title='mlogloss'),
    color=alt.Color('Dataset:N', legend=alt.Legend(title='Dataset'))
)
rule10 = alt.Chart(
    pd.DataFrame({'Round': [best_round]})
).mark_rule(color='#E53935', strokeDash=[6, 4], strokeWidth=2).encode(
    x='Round:Q'
)
label10 = alt.Chart(
    pd.DataFrame({'Round': [best_round + 2], 'Loss': [max(train_loss) * 0.92],
                  'text': [f'Best: {best_round}']})
).mark_text(align='left', fontSize=14, color='#E53935', fontStyle='italic').encode(
    x='Round:Q', y='Loss:Q', text='text:N'
)
chart10 = alt.layer(line10, rule10, label10).properties(
    title='Training Loss XGBoost — Log Loss per Boosting Round',
    width=520, height=300
)
chart10

Classification Report — XGBoost


,precision,recall,f1-score,support
Lancar,0.925300,0.956000,0.940400,1114.000000
Macet,0.917600,0.816800,0.864300,191.000000
Perhatian,0.877800,0.857600,0.867500,695.000000
accuracy,0.908500,0.908500,0.908500,0.908500
macro avg,0.906900,0.876800,0.890700,2000.000000
weighted avg,0.908000,0.908500,0.907800,2000.000000



=== Data Chart 9: Confusion Matrix — XGBoost ===


Prediksi,Lancar,Macet,Perhatian
Aktual,,,
Lancar,1065,0,49
Macet,1,156,34
Perhatian,85,14,596


  Total prediksi benar : 1,817 / 2,000
  Akurasi              : 0.9085


alt.LayerChart(...)


=== Data Chart 10: Training Loss — XGBoost ===
  Total boosting rounds : 193
  Best iteration        : 173
  Train loss (awal)     : 0.869160
  Train loss (akhir)    : 0.096727
  Val loss (awal)       : 0.872203
  Val loss (akhir)      : 0.240827
  Val loss (best)       : 0.239998


alt.LayerChart(...)

In [23]:
# ============================================================
# Feature Importance — XGBoost
# ============================================================

FEATURE_LABEL_MAP = {
    'umur':                       'Usia Nasabah',
    'lama_bekerja_tahun':         'Lama Bekerja (th)',
    'lama_di_pekerjaan_sekarang': 'Lama di Pekerjaan Skrg',
    'penghasilan_bulanan':        'Penghasilan Bulanan',
    'penghasilan_tambahan':       'Penghasilan Tambahan',
    'penghasilan_pasangan':       'Penghasilan Pasangan',
    'total_penghasilan':          'Total Penghasilan',
    'jumlah_tanggungan':          'Jumlah Tanggungan',
    'jumlah_anak':                'Jumlah Anak',
    'jumlah_pinjaman_diajukan':   'Pinjaman Diajukan',
    'sisa_pinjaman_lain':         'Sisa Pinjaman Lain',
    'total_eksposur_kredit':      'Total Eksposur Kredit',
    'tenor_bulan':                'Tenor (bulan)',
    'angsuran_bulanan':           'Angsuran Bulanan',
    'debt_to_income_ratio':       'DTI Ratio',
    'loan_to_income_ratio':       'LTI Ratio',
    'saldo_tabungan':             'Saldo Tabungan',
    'nilai_aset':                 'Nilai Aset',
    'rasio_aset_utang':           'Rasio Aset/Utang',
    'jumlah_kredit_sebelumnya':   'Kredit Sebelumnya',
    'jumlah_tunggakan':           'Jumlah Tunggakan',
    'hari_tunggakan_terlama':     'Hari Tunggakan Terlama',
    'lama_tinggal':               'Lama Tinggal',
    'jenis_kelamin':              'Jenis Kelamin',
    'status_pernikahan':          'Status Pernikahan',
    'pasangan_bekerja':           'Pasangan Bekerja',
    'pendidikan':                 'Pendidikan',
    'sektor_pekerjaan':           'Sektor Pekerjaan',
    'level_pekerjaan':            'Level Pekerjaan',
    'risiko_pekerjaan':           'Risiko Pekerjaan',
    'status_rumah':               'Status Rumah',
    'tujuan_pinjaman':            'Tujuan Pinjaman',
    'kategori_tujuan':            'Kategori Tujuan',
    'riwayat_kredit':             'Riwayat Kredit',
    'urban_rural':                'Urban/Rural',
    'wilayah_risiko':             'Wilayah Risiko',
}

importance   = xgb_model.feature_importances_
fi_df = pd.DataFrame({
    'Fitur':     FEATURE_COLS,
    'Label':     [FEATURE_LABEL_MAP.get(f, f) for f in FEATURE_COLS],
    'Importance': importance
})
fi_top20 = fi_df.nlargest(20, 'Importance').sort_values('Importance')

print('=== Data Chart 11: Top 20 Feature Importance — XGBoost (gain) ===')
for rank, (_, row) in enumerate(fi_top20.iloc[::-1].iterrows(), 1):
    print(f'  {rank:2d}. {row["Label"]:30s}  {row["Importance"]:.4f}')

# --- Chart 11: Horizontal bar feature importance ---
bar11 = alt.Chart(fi_top20).mark_bar().encode(
    y=alt.Y('Label:N', sort=None, title=None),
    x=alt.X('Importance:Q', title='Importance Score (gain)'),
    color=alt.Color('Importance:Q',
                    scale=alt.Scale(scheme='blues'),
                    legend=None),
    tooltip=['Label:N', alt.Tooltip('Importance:Q', format='.4f')]
)
text11 = bar11.mark_text(align='left', dx=3, fontSize=12).encode(
    text=alt.Text('Importance:Q', format='.4f')
)
chart11 = (bar11 + text11).properties(
    title='Top 20 Feature Importance — XGBoost (gain)',
    width=480, height=460
)
chart11

=== Data Chart 11: Top 20 Feature Importance — XGBoost (gain) ===
   1. DTI Ratio                       0.2179
   2. Hari Tunggakan Terlama          0.1674
   3. Jumlah Tunggakan                0.0924
   4. Riwayat Kredit                  0.0689
   5. Saldo Tabungan                  0.0336
   6. Penghasilan Bulanan             0.0320
   7. Total Penghasilan               0.0192
   8. Kredit Sebelumnya               0.0172
   9. Level Pekerjaan                 0.0137
  10. LTI Ratio                       0.0136
  11. Pinjaman Diajukan               0.0135
  12. Sisa Pinjaman Lain              0.0135
  13. Lama Tinggal                    0.0134
  14. Pendidikan                      0.0134
  15. Rasio Aset/Utang                0.0133
  16. Lama Bekerja (th)               0.0131
  17. Penghasilan Pasangan            0.0131
  18. Lama di Pekerjaan Skrg          0.0131
  19. Kategori Tujuan                 0.0131
  20. Pasangan Bekerja                0.0131


alt.LayerChart(...)

## 6. Kesimpulan & Temuan Penelitian

### 6.1 Ringkasan Benchmarking

Dari eksperimen ini, diperoleh hasil benchmarking ketiga model. XGBoost secara konsisten mengungguli baseline karena:
1. Kemampuan menangkap interaksi non-linear antar fitur
2. Built-in regularisasi (L1/L2) mencegah overfitting
3. Early stopping mencegah training berlebihan
4. Gradient boosting secara iteratif memperbaiki prediksi pada sampel sulit

### 6.2 Fitur Paling Prediktif

Berdasarkan feature importance XGBoost, fitur-fitur berikut paling berpengaruh:

| Rank | Fitur | Interpretasi |
|---|---|---|
| 1 | **Hari Tunggakan Terlama** | Indikator langsung riwayat keterlambatan |
| 2 | **DTI Ratio** | Beban utang relatif terhadap penghasilan |
| 3 | **Jumlah Tunggakan** | Frekuensi keterlambatan pembayaran |
| 4 | **Riwayat Kredit** | Rekam jejak kepatuhan pembayaran |
| 5 | **Saldo Tabungan** | Cadangan likuiditas nasabah |

### 6.3 Implikasi Praktis

- Model dapat digunakan untuk **pre-screening** aplikasi kredit secara otomatis
- Fitur dengan importance tinggi dapat menjadi **kriteria prioritas** dalam form aplikasi
- Perlu validasi dengan data nyata sebelum deployment produksi
- Kelas **Macet** (minoritas ~10%) perlu perhatian khusus — pertimbangkan class weighting atau oversampling

In [24]:
# ============================================================
# Simpan Artefak Model
# ============================================================
import os

os.makedirs('models_xgb', exist_ok=True)

# Simpan model XGBoost
xgb_model.save_model('models_xgb/xgb_credit.ubj')

# Simpan encoders kategorikal
joblib.dump(encoders, 'models_xgb/xgb_encoders.joblib')

# Simpan target encoder
joblib.dump(target_encoder, 'models_xgb/xgb_target_encoder.joblib')

# Verifikasi
import os
files = [
    ('synthetic_credit_10k.csv', 'Dataset sintetik'),
    ('models_xgb/xgb_credit.ubj', 'Model XGBoost'),
    ('models_xgb/xgb_encoders.joblib', 'Encoders kategorikal'),
    ('models_xgb/xgb_target_encoder.joblib', 'Target encoder'),
]

print('Artefak yang tersimpan:')
print('-' * 55)
for fpath, desc in files:
    size = os.path.getsize(fpath) / 1024 if os.path.exists(fpath) else 0
    status = 'OK' if os.path.exists(fpath) else 'MISSING'
    print(f'  [{status}] {desc:<30s}  {size:6.1f} KB')
    print(f'        {fpath}')

print('-' * 55)
print('Pipeline CLVS-ML selesai.')

Artefak yang tersimpan:
-------------------------------------------------------
  [OK] Dataset sintetik                3429.4 KB
        synthetic_credit_10k.csv
  [OK] Model XGBoost                   1432.6 KB
        models_xgb/xgb_credit.ubj
  [OK] Encoders kategorikal               3.4 KB
        models_xgb/xgb_encoders.joblib
  [OK] Target encoder                     0.5 KB
        models_xgb/xgb_target_encoder.joblib
-------------------------------------------------------
Pipeline CLVS-ML selesai.


In [25]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [28]:
mv /content/synthetic_credit_10k.csv /content/drive/MyDrive/'Colab Notebooks'/models_xgb